# Heritage Short: ACE-Step 1.5 score candidates (T4)
Pushed by `pipeline/music.py` via `pipeline/kaggle_job.py`. The JOBS cell lists the candidates:
caption (instruments + mood only), and bpm / key / time signature / duration as separate metadata.
Instrumental cues use the `[Instrumental]` lyric tag. float16 on the T4 (no bfloat16), CPU offload.
Writes `out/<cue>_c<n>.wav` and `timing.jsonl` to /kaggle/working.

In [ ]:
# @JOBS (replaced at push time)
JOBS = {"job_id": "local-test", "config": {}, "jobs": []}

In [ ]:
import json, os, sys, time, glob, hashlib, subprocess, traceback
T0 = time.time()
OUT = "/kaggle/working"
os.makedirs(f"{OUT}/out", exist_ok=True)
_log = open(f"{OUT}/timing.jsonl", "a")
def log(**kw):
    """One JSON line per attempt; pipeline/kaggle_job.py copies these into runs.jsonl."""
    _log.write(json.dumps({"job_id": JOBS["job_id"], **kw}) + "\n"); _log.flush(); print(kw)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.39.0", "soundfile"], check=True)

In [ ]:
import numpy as np, soundfile as sf, torch
from diffusers import AceStepPipeline

cfg = JOBS["config"]
pipe = AceStepPipeline.from_pretrained(cfg["model"], torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
log(id=None, event="setup", seconds=round(time.time() - T0, 1), gpu=torch.cuda.get_device_name(0),
    torch=torch.__version__)

In [ ]:
for job in JOBS["jobs"]:
    torch.cuda.reset_peak_memory_stats()
    t = time.time()
    try:
        audio = pipe(prompt=job["caption"], lyrics="[Instrumental]" if job["instrumental"] else "",
                     audio_duration=float(job["duration_s"]), bpm=job["bpm"], keyscale=job["keyscale"],
                     timesignature=job["timesignature"], num_inference_steps=job["steps"], shift=job["shift"],
                     generator=torch.Generator("cpu").manual_seed(job["seed"])).audios
        wav = audio[0].T.cpu().float().numpy()          # [samples, channels]
        if not np.isfinite(wav).all() or float(np.abs(wav).max()) < 1e-3:
            raise RuntimeError("NaN or silent audio (float16 overflow?)")
        sf.write(f"{OUT}/out/{job['id']}.wav", wav, pipe.sample_rate)
        log(id=job["id"], status="ok", attempt=1, seed=job["seed"], gpu_seconds=round(time.time() - t, 1),
            duration_s=round(len(wav) / pipe.sample_rate, 2), file=f"out/{job['id']}.wav",
            peak_vram_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))
    except Exception as e:
        log(id=job["id"], status="error", attempt=1, seed=job["seed"], gpu_seconds=round(time.time() - t, 1),
            error=repr(e)[:400])
        traceback.print_exc()
        torch.cuda.empty_cache()
log(id=None, event="done", seconds=round(time.time() - T0, 1))